In [ ]:
import pandas as pd
import numpy as np
from numpy.linalg import svd
import math
from sklearn.decomposition import NMF

# Read the data from the CSV file
data = pd.read_csv('MovieReviewMat.csv')

In [2]:
# Separate the first row (movie genres) and column headers
movie_genres = data.iloc[0, 1:].values.tolist()
movie_titles = data.columns.values[1:].tolist()

# Remove the first row from the data matrix
df = data.iloc[1:]
df = df.iloc[:,1:]

# Convert the data to numeric values (replace "NA" with NaN)
df = df.apply(pd.to_numeric, errors='coerce')

# Find the highest review, lowest review, and overall average review
highest_review = df.max().max()
lowest_review = df.min().min()
overall_average_review = np.nanmean(df).round(2)

print("Highest Review:", highest_review)
print("Lowest Review:", lowest_review)
print("Overall Average Review:", overall_average_review)

Highest Review: 5.0
Lowest Review: 0.5
Overall Average Review: 3.3


In [3]:
# Function to get the genre of movies with review score 5 for a given individual
def get_genre_title_with_review_score_of_5(individual_reviews, movie_genres, movie_titles, num_films=2):
    genre_with_score_of_5 = []
    title_with_score_of_5 = []
    count = 0
    for i, score in enumerate(individual_reviews):
        if score == 5:
            genre_with_score_of_5.append(movie_genres[i])
    for i, score in enumerate(individual_reviews):
        if score == 5:
            count += 1
            title_with_score_of_5.append(movie_titles[i])
            if count == num_films:
                break
    return genre_with_score_of_5, title_with_score_of_5

In [4]:
# Get the individual 1462 (indexed by 1460) and individual 45 (indexed by 43) reviews
individual_1462_reviews = df.iloc[1460]
individual_45_reviews = df.iloc[43]

individual_1462 = get_genre_title_with_review_score_of_5(individual_1462_reviews, movie_genres, movie_titles)
print("Genres of Movies Individual 1462 tends to give review scores of 5:", individual_1462[0])
print("Two Films Individual 1462 gave the highest possible score:", individual_1462[1], "\n")

# Examine individual 45's preferences
individual_45 = get_genre_title_with_review_score_of_5(individual_45_reviews, movie_genres, movie_titles)
print("Genres of Movies Individual 45 tends to give review scores of 5:", individual_45[0])
print("Two Films Individual 45 gave the highest possible score:", individual_45[1])

Genres of Movies Individual 1462 tends to give review scores of 5: ['Crime|Drama|Thriller', 'Action|Drama|Romance|War', 'Children|Drama', 'Drama|Romance', 'Drama|Romance', 'Drama', 'Drama|Musical', 'Children|Drama|Fantasy', 'Adventure|Drama', 'Drama|Romance', 'Action|Drama|Sci-Fi']
Two Films Individual 1462 gave the highest possible score: ['Taxi Driver (1976)', 'Rob Roy (1995)'] 

Genres of Movies Individual 45 tends to give review scores of 5: ['Comedy|Drama', 'Comedy|Crime', 'Comedy|Fantasy|Romance', 'Adventure|Comedy|Sci-Fi', 'Comedy|Drama']
Two Films Individual 45 gave the highest possible score: ['Jack (1996)', 'Sting, The (1973)']


PART D

In [ ]:
# Read the data from the CSV file
data = pd.read_csv('MovieReviewMat.csv')

# Remove the first row (movie genres) and set index to start from 0
df1 = data.iloc[1:, 1:].reset_index(drop=True)
df = df1.fillna(0)
df = df.apply(pd.to_numeric)
review_matrix = df.values

In [6]:
# Perform SVD
U, S, Vt = svd(review_matrix, full_matrices=False)

In [7]:
# Reconstruct the matrix using reduced rank K
K = 8
reconstructed_matrix = np.dot(U[:,:K] * S[:K], Vt[:K,:])

In [8]:
df2 = df1.apply(pd.to_numeric)

In [9]:
mask = np.isfinite(df2.values)

In [10]:
# Estimate user 45's review of the movie Jumanji (second column, indexed by 1)
user_45_review_jumanji = reconstructed_matrix[43, 1]

# Calculate the root mean squared error (RMSE) between true and reconstructed values
N = 1120381
rmse = np.sqrt(np.sum((df2.values[mask] - reconstructed_matrix[mask]) ** 2) / N)

print("Estimate of User 45's Review of Movie Jumanji:", user_45_review_jumanji.round(2))
print("Average Difference (RMSE) between True and Reconstructed Values:", round(rmse,2))

Estimate of User 45's Review of Movie Jumanji: 1.59
Average Difference (RMSE) between True and Reconstructed Values: 1.81


PART E

In [11]:
# Convert the data to a numeric matrix with 'NA' replaced by NaN
review_matrix = df1.apply(pd.to_numeric, errors='coerce').values

In [12]:
# Compute the column means and replace NaN (missing) values with the column mean
column_means = np.nanmean(review_matrix, axis=0)
column_means = np.where(np.isnan(column_means), 0, column_means)
review_matrix[np.isnan(review_matrix)] = np.take(column_means, np.isnan(review_matrix).nonzero()[1])

In [13]:
# Perform SVD
U, S, Vt = svd(review_matrix)

In [14]:
# Determine the number of "types" of individuals based on non-zero singular values (K)
K = 8
reconstructed_matrix = np.dot(U[:,:K] * S[:K], Vt[:K,:])

In [15]:
# Estimate user 45's review of the movie Jumanji (second column, indexed by 1)
user_45_review_jumanji = reconstructed_matrix[43, 1]

# Calculate the root mean squared error (RMSE) between true and reconstructed values
rmse = math.sqrt(np.sum((review_matrix[mask] - reconstructed_matrix[mask])**2) / N)

print("Estimate of User 45's Review of Movie Jumanji:", user_45_review_jumanji.round(2))
print("Average Difference (RMSE) between True and Reconstructed Values:", round(rmse,2))

Estimate of User 45's Review of Movie Jumanji: 2.45
Average Difference (RMSE) between True and Reconstructed Values: 0.8


PART F

In [16]:
# Remove the first row (movie genres) and set index to start from 0
review_matrix = df1.apply(pd.to_numeric, errors='coerce').values

In [17]:
# Compute the row means and replace NaN (missing) values with the row mean
row_means = np.nanmean(review_matrix, axis=1, keepdims=True)
review_matrix[np.isnan(review_matrix)] = np.take(row_means, np.isnan(review_matrix).nonzero()[0])

In [18]:
# Perform SVD
U, S, Vt = svd(review_matrix)

In [19]:
# Determine the number of "types" of individuals based on non-zero singular values (K)
K = 8
reconstructed_matrix = np.dot(U[:,:K] * S[:K], Vt[:K,:])

In [20]:
# Estimate user 45's review of the movie Jumanji (second column, indexed by 1)
user_45_review_jumanji = reconstructed_matrix[43, 1]

# Calculate the root mean squared error (RMSE) between true and reconstructed values
rmse = np.sqrt(np.sum((review_matrix[mask] - reconstructed_matrix[mask]) ** 2)/N)

print("Estimate of User 45's Review of Movie Jumanji:", user_45_review_jumanji.round(2))
print("Average Difference (RMSE) between True and Reconstructed Values:", round(rmse,3))

Estimate of User 45's Review of Movie Jumanji: 2.44
Average Difference (RMSE) between True and Reconstructed Values: 0.81


PART H

In [21]:
# Convert the data to a numeric matrix with 'NA' replaced by NaN
review_matrix = df1.apply(pd.to_numeric, errors='coerce').values

In [22]:
# Compute the column means and replace NaN (missing) values with the column mean
column_means = np.nanmean(review_matrix, axis=0)
column_means = np.where(np.isnan(column_means), 0, column_means)
review_matrix[np.isnan(review_matrix)] = np.take(column_means, np.isnan(review_matrix).nonzero()[1])

In [23]:
K = 8

# Perform NMF with column mean padding
nmf_model = NMF(n_components=K, init='random', random_state=0, max_iter=200)
W = nmf_model.fit_transform(review_matrix)
H = nmf_model.components_
reconstructed_matrix_nmf = np.dot(W, H)

C:\Users\emmanyel\AppData\Roaming\Python\Python310\site-packages\sklearn\decomposition\_nmf.py:1665: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


In [24]:
# Estimate user 45's review of the movie Jumanji (second column, indexed by 1)
user_45_review_jumanji_nmf = reconstructed_matrix_nmf[43, 1]

# Calculate the root mean squared error (RMSE) between true and reconstructed values for NMF
N = 1120381
rmse_nmf = np.sqrt(np.sum((review_matrix[mask] - reconstructed_matrix_nmf[mask]) ** 2) / N)

print("Estimate of User 45's Review of Movie Jumanji (NMF):", user_45_review_jumanji_nmf.round(2))
print("Average Difference (RMSE) between True and Reconstructed Values (NMF):", round(rmse,3))

Estimate of User 45's Review of Movie Jumanji (NMF): 2.49
Average Difference (RMSE) between True and Reconstructed Values (NMF): 0.81


PART J

In [25]:
# Set the value of K (number of latent factors)
K = 5

# Perform NMF with column mean padding
nmf_model = NMF(n_components=K, init='random', random_state=0, max_iter=200)
W = nmf_model.fit_transform(review_matrix)
H = nmf_model.components_
reconstructed_matrix_nmf = np.dot(W, H)

# Index of user 1462 (index starts from 0)
user_index = 1460

C:\Users\emmanyel\AppData\Roaming\Python\Python310\site-packages\sklearn\decomposition\_nmf.py:1665: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


In [26]:
find = df1.apply(pd.to_numeric, errors='coerce').values
find[np.isnan(find)] = 0.0

In [27]:
# Find movies user 1462 has not seen (rating = 0 in the review matrix)
unseen_movies_indices = np.where(find[user_index, :] == 0)[0]

In [28]:
# Get the predicted ratings for unseen movies by user 1462
predicted_ratings = reconstructed_matrix_nmf[user_index, unseen_movies_indices]

# Get the movie titles from the original dataset
movie_titles1 = data.columns[1:]

In [29]:
# Sort the unseen movies based on predicted ratings in descending order
sorted_indices = np.argsort(predicted_ratings)[::-1]
top_3_recommendations_indices = unseen_movies_indices[sorted_indices[:3]]
top_3_recommendations = movie_titles1[top_3_recommendations_indices]

print("Top 3 Movie Recommendations for User 1462:")
for i, movie in enumerate(top_3_recommendations, 1):
    print(f"{i}. {movie}")

Top 3 Movie Recommendations for User 1462:
1. Before Sunset (2004)
2. K-PAX (2001)
3. Nine to Five (a.k.a. 9 to 5) (1980)


PART K

In [ ]:
import pandas as pd
import numpy as np
from numpy.linalg import svd

# Read the data from the CSV file
data = pd.read_csv('MovieReviewMat.csv')
data = data.iloc[1:].reset_index(drop=True)
data = data.fillna(0)
data = data.apply(pd.to_numeric)

# Convert the data to a numeric matrix
review_matrix = data.iloc[:, 1:].values

# Set the value of K (number of latent factors)
K = 5

# Define the regularization penalty (λ)
lambda_value = 100

# Soft impute for 100 iterations
for _ in range(100):
    # Update missing values in the matrix with the soft-imputed values
    U, S, Vt = svd(review_matrix, full_matrices= False)
    sigma = np.maximum(S - lambda_value, 0)
    U = U[:, :K]
    sigma = sigma[:K]
    Vt = Vt[:K, :]
    reconstructed_matrix = U @np.diag(sigma) @ Vt
    X = np.where(np.isnan(S), reconstructed_matrix, S)

In [39]:
# Index of user 45 (index starts from 0)
user_index = 43

# Find movies user 45 has not seen (rating = 0 in the review matrix)
cx = data.iloc[:, 1:].values
unseen_movies_indices = np.where(cx[user_index, :] == 0)[0]

# Get the predicted ratings for unseen movies by user 45
predicted_ratings = reconstructed_matrix[user_index, unseen_movies_indices]

# Get the movie titles from the original dataset
movie_titles = data.columns[1:]

# Sort the unseen movies based on predicted ratings in descending order
sorted_indices = np.argsort(predicted_ratings)[::-1]
top_4_recommendations_indices = unseen_movies_indices[sorted_indices[:4]]
top_4_recommendations = movie_titles[top_4_recommendations_indices]

print("Top 4 Movie Recommendations for User 45 (Soft Impute with SVD):")
for i, movie in enumerate(top_4_recommendations, 1):
    print(f"{i}. {movie}")

Top 4 Movie Recommendations for User 45 (Soft Impute with SVD):
1. Ghost and Mrs. Muir, The (1947)
2. Killing Fields, The (1984)
3. M (1931)
4. 2001: A Space Odyssey (1968)


In [37]:
sorted(predicted_ratings[0:4], reverse=True)

[1.2925786416283709,
 0.27758134900656434,
 0.22394523191412966,
 -0.008090131433412957]